# Seed order experiments

This notebook tests how the ordering of seed integrals affects the cost of an IBP reduction, using `pyfeyngym.run_ibp_no_reordering` to find the cost in arithmetic operations without allowing the linear solver to reorder equations.

In [1]:
from pathlib import Path
from pprint import pprint

import pyfeyngym as pfg

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


We use the built-in massive-bubble Gym environment only to obtain the same list of seed candidates

In [2]:
target_integral = (6, 6)

env = pfg.IBPEnv(
    target_integral=list(target_integral),
    max_seed_propagator_power=6,
)

all_seeds = env.all_seed_candidates()
len(all_seeds), all_seeds

(48,
 [(1, 0),
  (0, 1),
  (0, 2),
  (1, 1),
  (2, 0),
  (3, 0),
  (1, 2),
  (2, 1),
  (0, 3),
  (4, 0),
  (3, 1),
  (2, 2),
  (1, 3),
  (0, 4),
  (0, 5),
  (5, 0),
  (3, 2),
  (4, 1),
  (1, 4),
  (2, 3),
  (5, 1),
  (4, 2),
  (3, 3),
  (2, 4),
  (1, 5),
  (0, 6),
  (6, 0),
  (3, 4),
  (4, 3),
  (1, 6),
  (2, 5),
  (6, 1),
  (5, 2),
  (6, 2),
  (5, 3),
  (3, 5),
  (4, 4),
  (2, 6),
  (5, 4),
  (4, 5),
  (3, 6),
  (6, 3),
  (4, 6),
  (6, 4),
  (5, 5),
  (6, 5),
  (5, 6),
  (6, 6)])

Set up the generic equation-generation and reduction inputs. The path lookup below works whether the notebook server was started from the repository root or from the `examples/` directory.

In [3]:
def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyfeyngym" / "examples" / "bubble-IBP").exists():
            return path
    raise FileNotFoundError("Could not find pyfeyngym/examples/bubble-IBP")


repo_root = find_repo_root()
ibp_file = repo_root / "pyfeyngym" / "examples" / "bubble-IBP"

m_vals = {"d": 1009, "m0": 1}
trivial_sectors = [0]
masters = [(1, 0), (0, 1), (1, 1)]
modulus = 2**31 - 1

eq_templates = pfg.gen_eq_templates(ibp_file, m_vals)
len(eq_templates)

2

The helper below generates equations from a chosen seed order, impose a prdefined variable order by `sort_integrals_desc`, and calls `run_ibp_no_reordering`.

In [4]:
def cost_from_seed_order(seeds, failure_cost=10**6):
    seed_op_eq_list, variables = pfg.gen_eqs(
        eq_templates,
        trivial_sectors,
        m_vals,
        seeds,
    )

    equations = [eq for _, _, eq in seed_op_eq_list]

    master_set = set(masters)
    variables = list(set(variables) | master_set | {target_integral})

    vars_sorted = pfg.sort_integrals_desc(variables)
    vars_ordered = (
        [v for v in vars_sorted if v not in master_set]
        + [v for v in vars_sorted if v in master_set]
    )

    reduction_complete, cost, n_eqs_used = pfg.run_ibp_no_reordering(
        equations,
        vars_ordered,
        target_integral,
        masters,
        modulus,
    )

    return {
        "reported_cost": cost if reduction_complete else failure_cost,
        "reduction_complete": reduction_complete,
        "raw_cost": cost,
        "n_eqs_used": n_eqs_used,
        "n_equations": len(equations),
        "n_variables": len(variables),
    }

First test the environment's default seed-candidate order.

In [5]:
cost_from_seed_order(all_seeds)

{'reported_cost': 1437,
 'reduction_complete': True,
 'raw_cost': 1437,
 'n_eqs_used': 92,
 'n_equations': 96,
 'n_variables': 74}

Now reverse the seed order. The equations are the same, but the fixed equation order changes.

In [6]:
all_seeds_reversed = list(reversed(all_seeds))
cost_from_seed_order(all_seeds_reversed)

{'reported_cost': 1707,
 'reduction_complete': True,
 'raw_cost': 1707,
 'n_eqs_used': 95,
 'n_equations': 96,
 'n_variables': 74}

An insufficient seed list does not complete the reduction. Here we report a high `failure_cost`, while also returning the raw solver cost accumulated before failure.

In [7]:
cost_from_seed_order(all_seeds[:40], failure_cost=10**6)

{'reported_cost': 1000000,
 'reduction_complete': False,
 'raw_cost': 1191,
 'n_eqs_used': 80,
 'n_equations': 80,
 'n_variables': 64}

Finally, sort seeds by a simple hand-written priority function.

In [8]:
def priority_func(seed):
    return seed[1] - seed[0]

priority_sorted_seeds = sorted(all_seeds, key=priority_func)
cost_from_seed_order(priority_sorted_seeds)

{'reported_cost': 1286,
 'reduction_complete': True,
 'raw_cost': 1286,
 'n_eqs_used': 56,
 'n_equations': 96,
 'n_variables': 74}

Collect the examples into one table for easier comparison.

In [9]:
experiments = {
    "default order": all_seeds,
    "reverse order": all_seeds_reversed,
    "first 40 seeds": all_seeds[:40],
    "priority order": priority_sorted_seeds,
}

results = {
    name: cost_from_seed_order(seeds)
    for name, seeds in experiments.items()
}

pprint(results)

{'default order': {'n_eqs_used': 92,
                   'n_equations': 96,
                   'n_variables': 74,
                   'raw_cost': 1437,
                   'reduction_complete': True,
                   'reported_cost': 1437},
 'first 40 seeds': {'n_eqs_used': 80,
                    'n_equations': 80,
                    'n_variables': 64,
                    'raw_cost': 1191,
                    'reduction_complete': False,
                    'reported_cost': 1000000},
 'priority order': {'n_eqs_used': 56,
                    'n_equations': 96,
                    'n_variables': 74,
                    'raw_cost': 1286,
                    'reduction_complete': True,
                    'reported_cost': 1286},
 'reverse order': {'n_eqs_used': 95,
                   'n_equations': 96,
                   'n_variables': 74,
                   'raw_cost': 1707,
                   'reduction_complete': True,
                   'reported_cost': 1707}}
